A script which creates a nested dictionary in which the final value at every nested key is a vector containing the processed timeseries for a specific trial, for channel, chromophore and participant, using 'fNIRSData.storeChannelData' <br>

Stored in the current working directory. 

Inputs:<br>
    &emsp;**datapath**:   directory containing the relevant timeseries matrices for all participants<br>
            &emsp;&emsp;filenames should be in the form: 'xxxx_y_zz' where<br>
            &emsp;&emsp;&emsp;    - xxxx is the participant number e.g. 0123<br>
            &emsp;&emsp;&emsp;    - y is the chromophore (1 = HbO, 2 = HbR, 3 = HbT)<br>
            &emsp;&emsp;&emsp;    - zz is the channel number (for BRIGHT: 1-34)<br>
    &emsp; **chrom**: the chromophore for which the the data is to be collected (1 = HbO, 2 = HbR, 3 = HbT)<br>
    &emsp; **channs**: the number of channels in the array<br>
    &emsp; **trials**: the number of trials in the experimental paradigm/trials for which data is to be collected<br>

Output:<br>
&emsp;A nested dictionary with key1 = participant ID (integer); key 2 = chromophore (1=HbO, 2=HbR, 3=HbT); key3 = Channel number; Value for key3: matrix containing all relevant timeseries for the relevant participant, chromophore and channel.

SB 27/02/23
        


First, import the relevant packages

In [51]:
import scipy.io
import os
import statistics
import pickle
import fNIRSData
import importlib


Give the destination directory for the dictionary and move working directory here:

In [52]:
workingDir = '/Users/sambe/Documents/Python_Projects/FDA'
os.chdir(workingDir)

Define the name of the directory containing data and the filename for the output dictionary:

In [53]:
# CHANGE DATAPATH & FILENAME FOR RELEVANT FOLDER!!!!
datapath = "/Users/sambe/Data/BRIGHT/UK/12mo/channelData"
outputFilename = 'dataRaw/homer2UK12moDataDictionary.pkl'

Find the participant IDs, number of chromophores used, number of channels in the array, in order to index the correct number of variables when storing the data:

In [54]:
##find participant IDs for indexing/keys in dictionary
#find filenames in directory (excl. README file and hidden files)
filenames = [f for f in os.listdir(datapath) if not (f.startswith('.') or f.startswith('R') or f.startswith('D'))]

#create a list of the unique participant IDs in the directory
partIDs = []
for filename in filenames:
    partIDs.append(filename[0:4])
partIDs = list(set(partIDs))

##find number of chromophores for indexing/keys in dictionary
#find chromophore numbers
chromNums = []
for filename in filenames:
    chromNums.append(filename[5:6])
# create set of chromophore numbers as integers
chromNums = list(set([int(chromNum) for chromNum in chromNums]))

##find number of channels for indexing/keys in dictionary
#find channel numbers
chanNums = []
for filename in filenames:
    chanNums.append(filename[7:])
# create set of channel numbers as integers, also removing the '.mat' suffix
chanNums = list(set([int(chanNum[:-4]) for chanNum in chanNums]))

del(filename)


Change directory, loop through data and store in the nested dictionary:

In [55]:
#change working directory
os.chdir(datapath)

# code in here occurs within the data directory
#create overriding nested dictionary to contain all data
dataDict = dict()
# loop through participants
for part in partIDs:
    #create integer version of participant number for storefNIRSData function
    partInt = int(part)
    #create a dictionary
    partDict = dict()
    #loop through chromophores
    for chrom in chromNums:
        #create the chromophore sub dictionary containing channel data
        chromDict = dict()
        #loop through channels
        for chann in chanNums:
            #create channel sub-dictionary containing all trial timeseries as key:value pairs
            chanDict = fNIRSData.storeChannelData(datapath, partInt, chrom, chann)
            #add the channel sub-dictionary dictionary to the chromophore dictionary
            chromDict[chann] = chanDict
        #add the chomophore sub-dictionary to the participant dictionary
        partDict[chrom] = chromDict
    dataDict[partInt] = partDict

#clear variables
del(filenames, part, partDict, partIDs, chanDict, chann, chanNums, chrom, chromDict, chromNums, partInt)

Move back to original directory and save the dictionary:


In [56]:
#move wd back to original 
os.chdir(workingDir)

#save the participant's nested dictionary
with open(outputFilename, 'wb') as f:
    pickle.dump(dataDict, f)

del(f, datapath, workingDir, dataDict)